# Construção da Camada Gold

A camada Gold é responsável por disponibilizar dados orientados ao consumo analítico e à tomada de decisão.

Serão construídas três tabelas Gold:

- `gold.acompanhamento_metas`
- `gold.resumo_brasil`
- `gold.indicadores_rede`

# Configuração

## Bibliotecas

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Construção da Camada Gold

Cria o schema

In [0]:
gold_schema = "gold"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {gold_schema}"
)

print(f"Schema '{gold_schema}' disponível.")

## 01. Gold - Acompanhamento das Metas de Alfabetização

Esta tabela tem como objetivo comparar o resultado efetivamente observado da taxa de alfabetização com a meta estabelecida para o mesmo ano.

A tabela Silver de metas possui dois conceitos temporais:

- `ano_referencia`: ano em que o resultado foi observado;
- `ano_meta`: ano ao qual determinada meta se refere.

Para avaliar o atingimento da meta, são considerados os registros em que `ano_referencia = ano_meta`.

A tabela contempla os níveis geográficos:

- Município;
- UF;
- Brasil.

Além dos valores observados e das metas, são calculados:

- diferença em pontos percentuais para a meta;
- percentual de atingimento da meta;
- indicador de atingimento;
- status do resultado.

Valores ausentes são preservados e classificados separadamente, evitando que ausência de informação seja interpretada como resultado igual a zero.

### Manipulações

In [0]:
df_metas = spark.table("silver.metas")

df_acompanhamento_metas = (
    df_metas
    
    # Filtrando -> Meta do mesmo ano das observações, para comparação justa e atualizada
    .filter(F.col("ano_referencia") == F.col("ano_meta"))

    # Calculando -> Gap entre taxa de alfabetização e meta, e percentual de atingimento da meta
    .withColumn(
        "gap_meta_pp",
        F.round(
            F.col("taxa_alfabetizacao_referencia") - F.col("meta_alfabetizacao"), 2
        ),
    )

    # Calculando -> Percentual de atingimento da meta
    .withColumn(
        "percentual_atingimento_meta",
        F.when(
            F.col("meta_alfabetizacao") > 0,
            F.round(
                (F.col("taxa_alfabetizacao_referencia") / F.col("meta_alfabetizacao"))
                * 100,
                2,
            ),
        ),
    )

    # Flag binária de atingimento da meta, para facilitar agregações
    .withColumn(
        "flag_atingiu_meta",
        F.when(
            F.col("taxa_alfabetizacao_referencia").isNull()
            | F.col("meta_alfabetizacao").isNull(),
            F.lit(None).cast("int"),
        )
        .when(
            F.col("taxa_alfabetizacao_referencia") >= F.col("meta_alfabetizacao"),
            F.lit(1),
        )
        .otherwise(F.lit(0)),
    )

    # Criando status descritivo pra meta
    .withColumn(
        "status_meta",
        F.when(F.col("taxa_alfabetizacao_referencia").isNull(), "SEM RESULTADO")
        .when(F.col("meta_alfabetizacao").isNull(), "SEM META")
        .when(
            F.col("taxa_alfabetizacao_referencia") >= F.col("meta_alfabetizacao"),
            "ATINGIU A META",
        )
        .otherwise("NÃO ATINGIU A META"),
    )

    # Selecionando, ordenando e renomeando algumas colunas
    .select(
        F.col("ano_referencia").alias("ano"),
        "nivel_geografico",
        "id_geografia",
        "rede",
        F.col("taxa_alfabetizacao_referencia").alias("taxa_alfabetizacao"),
        "meta_alfabetizacao",
        "gap_meta_pp",
        "percentual_atingimento_meta",
        "flag_atingiu_meta",
        "status_meta",
        F.col("percentual_participacao_referencia").alias("percentual_participacao"),
        F.col("nivel_alfabetizacao_referencia").alias("nivel_alfabetizacao"),
        "_ingestion_timestamp",
        "_source_system",
        "_source_table",
    )

    # Adicionando metadados
    .withColumn("_gold_processed_at", F.current_timestamp())
)


# Persistindo a tabela em formato delta
(
    df_acompanhamento_metas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.acompanhamento_metas")
)

### Visões

Schema e quantidade de registros

In [0]:
df_gold_metas = spark.table("gold.acompanhamento_metas")

print(f"Quantidade de registros: " f"{df_gold_metas.count():,}")

df_gold_metas.printSchema()

Percentual de atingimento por ano e nível geográfico

In [0]:
display(
    df_gold_metas
    .groupBy("ano", "nivel_geografico")
    .agg(
        F.count("*").alias("total_regioes"),
        F.sum("flag_atingiu_meta").alias("regioes_que_atingiram"),
        F.round(F.avg("flag_atingiu_meta") * 100, 2).alias("percentual_que_atingiu"),
    )
    .orderBy("ano", "nivel_geografico")
)

## 02. Gold - Indicadores por Rede de Ensino

Esta tabela consolida os dados individuais da camada `silver.alunos` em indicadores analíticos por ano e rede de ensino.

Os registros individuais de alunos são agregados por `ano` e `rede`.

As redes ausentes em determinado ano não são criadas artificialmente, preservando a cobertura original da fonte.

São calculados indicadores de:

- quantidade total de alunos;
- presença;
- preenchimento da prova;
- alfabetização entre alunos avaliados;
- alfabetização ponderada pelo peso amostral;
- proficiência média.

A rede Privada possui baixa quantidade de registros em 2024, portanto seus indicadores devem ser interpretados com cautela.

In [0]:
df_alunos = spark.table("silver.alunos")

df_indicadores_rede = (
    df_alunos

    # Agregações iniciais: Quantidade de alunos, presença, provas preenchidas, alfabetizados e proficiÊncia
    .groupBy("ano", "rede", "rede_descricao")
    .agg(
        F.count("*").alias("total_alunos"),
        F.sum(F.when(F.col("presenca") == "1", 1).otherwise(0)).alias("alunos_presentes"),
        F.sum(F.when(F.col("preenchimento_caderno") == "1", 1).otherwise(0)).alias("provas_preenchidas"),
        F.sum(F.when(
            (F.col("preenchimento_caderno") == "1")
            & (F.col("alfabetizado") == "1"), 1).otherwise(0)).alias("alunos_alfabetizados"),
        
        F.avg("proficiencia").alias("media_proficiencia"),

        F.sum(F.when(F.col("preenchimento_caderno") == "1", F.col("peso_aluno")).otherwise(0)).alias("peso_total_avaliados"),
        F.sum(
            F.when(
                (F.col("preenchimento_caderno") == "1")
                & (F.col("alfabetizado") == "1"),
                F.col("peso_aluno"),
            ).otherwise(0)
        ).alias("peso_alfabetizados"),
    )

    # Cálculo das taxas: Presença, preenchimento da prova, alfabetização
    .withColumn(
        "taxa_presenca_pct",
        F.round(F.col("alunos_presentes") / F.col("total_alunos") * 100, 2),
    )
    .withColumn(
        "taxa_preenchimento_pct",
        F.round(F.col("provas_preenchidas") / F.col("total_alunos") * 100, 2),
    )
    .withColumn(
        "taxa_preenchimento_presentes_pct",
        F.when(
            F.col("alunos_presentes") > 0,
            F.round(F.col("provas_preenchidas") / F.col("alunos_presentes") * 100, 2),
        ),
    )
    .withColumn(
        "taxa_alfabetizacao_avaliados_pct",
        F.when(
            F.col("provas_preenchidas") > 0,
            F.round(
                F.col("alunos_alfabetizados") / F.col("provas_preenchidas") * 100, 2
            ),
        ),
    )
    .withColumn(
        "taxa_alfabetizacao_ponderada_pct",
        F.when(
            F.col("peso_total_avaliados") > 0,
            F.round(
                F.col("peso_alfabetizados") / F.col("peso_total_avaliados") * 100, 2
            ),
        ),
    )

    # Organizando e ordenando as colunas do dataframe final
    .select(
        "ano",
        "rede",
        "rede_descricao",
        "total_alunos",
        "alunos_presentes",
        "taxa_presenca_pct",
        "provas_preenchidas",
        "taxa_preenchimento_pct",
        "taxa_preenchimento_presentes_pct",
        "alunos_alfabetizados",
        "taxa_alfabetizacao_avaliados_pct",
        "taxa_alfabetizacao_ponderada_pct",
        F.round("media_proficiencia", 2).alias("media_proficiencia"),
    )

    # Metadados
    .withColumn("_gold_processed_at", F.current_timestamp())
)


# Persistindo em Delta
(
    df_indicadores_rede.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "gold.indicadores_rede"
    )
)

In [0]:
df_indicadores_rede.display()